# 02 - Statistical Analysis

The exploratory analysis identified several patterns in daily bike rental demand. In this notebook, we investigate some of these patterns using formal statistical methods.

The aim is to determine whether the observed differences in rental demand are statistically significant rather than simply the result of variation within the sample.

The analysis is divided into four stages:

1. Formulate statistical questions
2. Perform hypothesis tests
3. Calculate confidence intervals
4. Compare rental demand across weather conditions using ANOVA and post-hoc tests

A significance level of α = 0.05 will be used throughout the analysis.


## 1. Working Days

### Statistical question

Does the mean number of daily bike rentals differ between working and non-working days?

### Hypotheses

**H₀:** There is no difference in mean daily rentals between working and non-working days.

**H₁:** There is a difference in mean daily rentals between working and non-working days.

**Significance level:** α = 0.05

Because we are comparing the means of two independent groups, we use an independent two-sample t-test. Welch's version of the test is used because it does not assume equal population variances.

In [1]:
import pandas as pd

data_days = pd.read_csv("../data/processed/data_days.csv", parse_dates=["dteday"])

In [2]:
#checking correct load
data_days.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,weather,working_label
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985,Mist / Cloudy,Non-working day
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801,Mist / Cloudy,Non-working day
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349,Clear,Working day
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562,Clear,Working day
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600,Clear,Working day


In [3]:
working_days = data_days[data_days.workingday == 1]
non_working_days = data_days[data_days.workingday == 0]

In [4]:
from scipy import stats

In [5]:
stats.ttest_ind(
    working_days["cnt"],
    non_working_days["cnt"],
    equal_var=False
)

TtestResult(statistic=np.float64(1.6013720686967898), pvalue=np.float64(0.11005737827017054), df=np.float64(413.93578279295593))

### Result

The Welch two-sample t-test gives a p-value of 0.110.

Since p > 0.05, we fail to reject the null hypothesis. There is not sufficient statistical evidence to conclude that mean daily bike rentals differ between working and non-working days.

## 2. Differences Between Years

### Statistical question

Does the mean number of daily bike rentals differ between 2011 and 2012?

### Hypotheses

**H₀:** There is no difference in mean daily rentals between 2011 and 2012.

**H₁:** There is a difference in mean daily rentals between 2011 and 2012.

**Significance level:** α = 0.05

As with the previous comparison, we use Welch's independent two-sample t-test.

In [6]:
#testing for the difference in the two years

stats.ttest_ind(
    data_days.loc[data_days.yr == 0, "cnt"],
    data_days.loc[data_days.yr == 1, "cnt"],
    equal_var=False
)

TtestResult(statistic=np.float64(-18.57770740475173), pvalue=np.float64(1.0484385153559308e-62), df=np.float64(685.4970138571425))

### Result

The Welch two-sample t-test gives a p-value of approximately 1.05 × 10⁻⁶².

Since p < 0.05, we reject the null hypothesis. There is very strong statistical evidence that mean daily bike rental demand differs between 2011 and 2012.

In [7]:
data_days.groupby("yr")["cnt"].mean()

yr
0    3405.761644
1    5599.934426
Name: cnt, dtype: float64

## 3. Weather Conditions

### Statistical question

Does mean daily bike rental demand differ between the different weather situations?

There are three weather groups in the dataset, so instead of performing multiple t-tests initially, we first use a one-way ANOVA.

### Hypotheses

**H₀:** The mean daily rental count is the same across all weather situations.

**H₁:** At least one weather situation has a different mean daily rental count.

**Significance level:** α = 0.05

### One-way ANOVA

A one-way ANOVA is used to test whether there is evidence that the mean rental count differs across the three weather groups.

In [8]:
stats.f_oneway(data_days[data_days["weathersit"] == 1]["cnt"], data_days[data_days["weathersit"] == 2]["cnt"], data_days[data_days["weathersit"] == 3]["cnt"])

F_onewayResult(statistic=np.float64(40.06604492024684), pvalue=np.float64(3.106317270053755e-17))

### Result

The one-way ANOVA gives an F-statistic of 40.07 and a p-value of approximately 3.11 × 10⁻¹⁷.

Since p < 0.05, we reject the null hypothesis. There is strong evidence that mean daily rental demand is not the same across all three weather situations.

However, ANOVA only tells us that at least one group differs. It does not tell us which groups differ from one another, so we perform pairwise post-hoc comparisons.

### Weather 1 vs Weather 3


In [9]:
test_13 = stats.ttest_ind(
    data_days.loc[data_days["weathersit" ] == 3, "cnt"],
    data_days.loc[data_days["weathersit" ] == 1, "cnt"],
    equal_var=False
)


The p-value is below 0.05, so there is evidence of a difference in mean daily rental demand between clear weather conditions (Weather 1) and the worst weather conditions (Weather 3).

### Weather 2 vs Weather 3

In [10]:
test_23 = stats.ttest_ind(
    data_days.loc[data_days["weathersit" ] == 3, "cnt"],
    data_days.loc[data_days["weathersit" ] == 2, "cnt"],
    equal_var=False
)


The p-value is below 0.05, so there is evidence of a difference in mean daily rental demand between mist/cloudy conditions (Weather 2) and the worst weather conditions (Weather 3).

### Weather 1 vs Weather 2

In [11]:
test_12 = stats.ttest_ind(
    data_days.loc[data_days["weathersit"] == 2, "cnt"],
    data_days.loc[data_days["weathersit"] == 1, "cnt"],
    equal_var=False
)


The p-value is below 0.05, so there is evidence of a difference in mean daily rental demand between clear conditions (Weather 1) and mist/cloudy conditions (Weather 2).

In [12]:
p_values = [test_12.pvalue, test_13.pvalue, test_23.pvalue]

In [13]:
from statsmodels.stats.multitest import multipletests
reject, p_adjusted, _, _ = multipletests(
    p_values, 
    alpha=0.05,
    method="holm"
)

In [14]:
posthoc_results = pd.DataFrame({
    "Comparison": [
        "Weather 1 vs 2",
        "Weather 1 vs 3",
        "Weather 2 vs 3"
    ],
    "Raw p-value": p_values,
    "Holm-adjusted p-value": p_adjusted,
    "Significant": reject
})

posthoc_results

,Comparison,Raw p-value,Holm-adjusted p-value,Significant
0,Weather 1 vs 2,1.033489e-08,2.066978e-08,True
1,Weather 1 vs 3,8.830918e-11,2.649275e-10,True
2,Weather 2 vs 3,2.996147e-08,2.996147e-08,True


### Post-hoc results

Because three pairwise comparisons are being performed, the Holm correction is used to control the family-wise error rate.

All three Holm-adjusted p-values are below α = 0.05. Therefore, there is evidence of a difference in mean daily rental demand between every pair of weather situations.

## 4. Confidence Intervals

In [15]:
weather_1 = data_days.loc[data_days["weathersit"] == 1, "cnt"]

mean = weather_1.mean()
sem = stats.sem(weather_1)
n = len(weather_1)

stats.t.interval(
    0.95,
    df=n-1,
    loc=mean,
    scale=sem
)

(np.float64(4705.139481558728), np.float64(5048.432872652935))

For clear weather days (Weather 1), the 95% confidence interval for mean daily bike rentals is between 4705 and 5048 rentals.

In [16]:
weather_2 = data_days.loc[data_days["weathersit"] == 2, "cnt"]
sem_2 = stats.sem(weather_2)
mean_2 = weather_2.mean()
n = len(weather_2)

stats.t.interval(
    0.95, 
    df=n-1, 
    loc=mean_2,
    scale=sem_2
)

(np.float64(3809.133668388691), np.float64(4262.5910279675845))

For mist/cloudy days (Weather 2), the 95% confidence interval for mean daily bike rentals is between 3809 and 4263 rentals.

In [17]:
weather_3 = data_days.loc[data_days["weathersit"] == 3, "cnt"]
mean = weather_3.mean()
n = len(weather_3)
sem = stats.sem(weather_3)

stats.t.interval(
    0.95, 
    df = n-1,
    loc = mean, 
    scale = sem
)

(np.float64(1238.7150018910093), np.float64(2367.856426680419))

For the worst weather conditions (Weather 3), the 95% confidence interval for mean daily bike rentals is between 1239 and 2368 rentals.

In [18]:
results = []

for weather in [1, 2, 3]:
    data = data_days.loc[data_days["weathersit"] == weather, "cnt"]
    
    mean = data.mean()
    sem = stats.sem(data)
    n = len(data)
    
    lower, upper = stats.t.interval(
        0.95,
        df=n-1,
        loc=mean,
        scale=sem
    )
    
    results.append({
        "Weather": weather,
        "Mean": mean,
        "CI Lower": lower,
        "CI Upper": upper
    })

ci_table = pd.DataFrame(results)
ci_table

,Weather,Mean,CI Lower,CI Upper
0,1,4876.786177,4705.139482,5048.432873
1,2,4035.862348,3809.133668,4262.591028
2,3,1803.285714,1238.715002,2367.856427


The confidence intervals provide estimates of the mean daily rental demand for each weather situation.

The intervals are clearly separated, which is consistent with the differences identified by the post-hoc hypothesis tests. The interval for Weather 3 is substantially wider than the others, reflecting the smaller number of observations in this group.

## 5. Conclusions

The statistical analysis provides evidence for several differences in daily bike rental demand.

- **Working days:** We fail to reject H₀. There is not sufficient statistical evidence of a difference in mean rental demand between working and non-working days.
- **Year:** We reject H₀. There is very strong statistical evidence that mean rental demand differs between 2011 and 2012.
- **Weather:** The ANOVA indicates that mean rental demand differs across weather situations. The Holm-adjusted post-hoc tests show significant differences between all three pairs of weather conditions.
- **Confidence intervals:** Mean rental demand is highest under clear conditions and lowest under the worst weather conditions. The confidence interval for the worst weather group is wider because this group contains fewer observations.

Overall, the statistical analysis supports several of the patterns identified during the exploratory analysis, particularly the differences associated with year and weather conditions.

## 6. Limitations

The analysis identifies statistical differences and associations but does not establish causal relationships.

The observations also contain temporal structure, meaning that daily rental counts may not be completely independent of nearby observations. Additionally, the weather groups are uneven in size, with substantially fewer observations in the worst weather category.